In [1]:
import numpy as np
import pandas as pd
import os
import torch

from scipy.signal import resample, butter, filtfilt, iirnotch
from scipy import stats
import warnings
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from pathlib import Path
import pickle
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
import h5py
from helper_functions import _load_model, notch_filter, zscore_signals
from PhysioNetChallenge2021_model import NN
from LSTM_model import *
warnings.filterwarnings("ignore", message="Creating a tensor from a list of numpy.ndarrays is extremely slow")
from model_pipeline import *

cuda:0


/mnt/beegfs/labs/cliffordlab/data/caisr_psg/CVDSleepPrediction/helper_functions.py:19: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  input = pickle.load(handle)


Model for embeddings extraction loaded ...
Model for CVD prediction loaded ...


In [2]:
### Create list of example data in .h5 format
path_data = 'data_example'
files = os.listdir(path_data)
files = [os.path.join(path_data, f)for f in files]

In [3]:
df_results = pd.DataFrame()
# Iterate through the example data
for i in range(len(files)):
    file_name = files[i]
    basename = os.path.basename(file_name)
    with h5py.File(file_name, 'r') as f:
        anot = f['annotations'] # `[:]` reads the entire dataset
        stage = anot['stage'][:][:,0] # extract sleep staging
        signals = f['signals']
        keys_channels = list(signals.keys())
        
        # Check for ECG channel
        if 'ecg' in keys_channels:
            recording = signals['ecg'][:][:,0]
        else:
            continue
            
    #####################
    ##### INFERENCE #####    
    #####################
    model_output = model_pipeline(recording, fs_original, stage = stage)
    #####################
    #####################
    
    # Extract probabilities
    af_probability = model_output['AF']
    stroke_probability = model_output['STROKE']
    mi_probability = model_output['MI']
    hf_probability = model_output['HF']
    death_probability = model_output['DEATH']
    
    print(f'Recording: {i} / {len(files)} : {basename}')
    print(f'CNN Future AF probability: {af_probability:.2f}')
    print(f'CNN Future Stroke probability: {stroke_probability:.2f}')
    print(f'CNN Future MI probability: {mi_probability:.2f}')
    print(f'CNN Future HF probability: {hf_probability:.2f}')
    print(f'CNN Future Death probability: {death_probability:.2f}')

    df_results.loc[i,'name'] = basename
    df_results.loc[i,'AF'] = af_probability
    df_results.loc[i, 'Stroke'] = stroke_probability
    df_results.loc[i, 'MI'] = mi_probability
    df_results.loc[i, 'HF'] = hf_probability
    df_results.loc[i, 'Death'] = death_probability
    print('-----'*10)
df_results.to_csv('results/cvd_future_prediction.csv', index = False)

Recording: 0 / 5 : sub-S0001111211287_ses-1.h5
CNN Future AF probability: 0.10
CNN Future Stroke probability: 0.13
CNN Future MI probability: 0.04
CNN Future HF probability: 0.12
CNN Future Death probability: 0.08
--------------------------------------------------
Recording: 1 / 5 : sub-S0001111189075_ses-1.h5
CNN Future AF probability: 0.02
CNN Future Stroke probability: 0.05
CNN Future MI probability: 0.01
CNN Future HF probability: 0.03
CNN Future Death probability: 0.01
--------------------------------------------------
Recording: 2 / 5 : sub-S0001111211297_ses-1.h5
CNN Future AF probability: 0.18
CNN Future Stroke probability: 0.12
CNN Future MI probability: 0.04
CNN Future HF probability: 0.14
CNN Future Death probability: 0.05
--------------------------------------------------
Recording: 3 / 5 : sub-S0001111189359_ses-1.h5
CNN Future AF probability: 0.05
CNN Future Stroke probability: 0.07
CNN Future MI probability: 0.02
CNN Future HF probability: 0.03
CNN Future Death probabili